In [29]:
import os
import shutil
import random

print("--- TIẾN HÀNH DÒ ĐƯỜNG DẪN THỰC TẾ VÀ TRÍCH XUẤT CCDDF ---")

# 1. Tự động quét toàn bộ vùng Input để tìm chính xác thư mục chứa ảnh của bộ 140k
source_base_140k = None

for root, dirs, files in os.walk('/kaggle/input'):
    # Tìm thư mục nào có tên là 'train' và bên trong nó có chứa thư mục 'fake'
    if root.endswith('train') and 'fake' in dirs:
        source_base_140k = root
        break

if source_base_140k is None:
    raise FileNotFoundError("Hệ thống vẫn chưa nhận diện được bộ 140k. Bạn hãy bấm nút 'Refresh' (vòng tròn mũi tên) ở góc phải mục Input xem sao!")

# Thiết lập chuẩn xác thư mục Fake và Real nguồn dựa trên kết quả dò được
fake_dir = os.path.join(source_base_140k, 'fake')
real_dir = os.path.join(source_base_140k, 'real')

print("👉 Đường dẫn nguồn thực tế tìm thấy:", source_base_140k)
print("-> Thư mục Fake nguồn:", fake_dir)
print("-> Thư mục Real nguồn:", real_dir)

# 2. Khởi tạo cấu trúc cây thư mục CCDDF tại vùng đầu ra /kaggle/working/
base_ccddf_dir = '/kaggle/working/ccddf_dataset/'
splits = ['train', 'valid', 'test']
classes = ['real', 'fake']

for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(base_ccddf_dir, split, cls), exist_ok=True)

# 3. Quét danh sách tệp ảnh hợp lệ
all_fake_images = [f for f in os.listdir(fake_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
all_real_images = [f for f in os.listdir(real_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

# Xáo trộn dữ liệu ngẫu nhiên
random.seed(42)
random.shuffle(all_fake_images)
random.shuffle(all_real_images)

# Phân bổ số lượng ảnh gốc (Train: 8000, Valid: 1700, Test: 300)
train_count, val_count, test_count = 8000, 1700, 300 

def split_and_copy(img_list, source_directory, class_name):
    for img in img_list[:train_count]:
        shutil.copy(os.path.join(source_directory, img), os.path.join(base_ccddf_dir, 'train', class_name, img))
    for img in img_list[train_count : train_count + val_count]:
        shutil.copy(os.path.join(source_directory, img), os.path.join(base_ccddf_dir, 'valid', class_name, img))
    for img in img_list[train_count + val_count : train_count + val_count + test_count]:
        shutil.copy(os.path.join(source_directory, img), os.path.join(base_ccddf_dir, 'test', class_name, img))

print("\n👉 Đang tiến hành sao chép file dữ liệu ngầm, vui lòng đợi từ 1 - 2 phút...")
split_and_copy(all_fake_images, fake_dir, 'fake')
print("Đã phân tách cấu trúc nguồn ảnh giả lập (Fake) xong.")
split_and_copy(all_real_images, real_dir, 'real')
print("Đã phân tách cấu trúc nguồn ảnh thực tế (Real) xong.")
print("\n✨ Hoàn thành! Tập dữ liệu CCDDF của bạn đã tạo xong tại:", base_ccddf_dir)

--- TIẾN HÀNH DÒ ĐƯỜNG DẪN THỰC TẾ VÀ TRÍCH XUẤT CCDDF ---
👉 Đường dẫn nguồn thực tế tìm thấy: /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train
-> Thư mục Fake nguồn: /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/fake
-> Thư mục Real nguồn: /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/real

👉 Đang tiến hành sao chép file dữ liệu ngầm, vui lòng đợi từ 1 - 2 phút...
Đã phân tách cấu trúc nguồn ảnh giả lập (Fake) xong.
Đã phân tách cấu trúc nguồn ảnh thực tế (Real) xong.

✨ Hoàn thành! Tập dữ liệu CCDDF của bạn đã tạo xong tại: /kaggle/working/ccddf_dataset/


In [30]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH_SIZE = 128  # Cấu hình chuẩn bắt buộc cho CCDDF

# Tăng cường dữ liệu (Data Augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30, width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2, horizontal_flip=True,
    brightness_range=[0.8, 1.2], fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

# Đọc dữ liệu từ thư mục CCDDF đã được tạo sẵn
train_gen = train_datagen.flow_from_directory(
    '/kaggle/working/ccddf_dataset/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_gen = val_datagen.flow_from_directory(
    '/kaggle/working/ccddf_dataset/valid',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

Found 16000 images belonging to 2 classes.
Found 3400 images belonging to 2 classes.


In [33]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_cnn_block(x, filters, dropout_rate=0.25):
    """
    Trích xuất nguyên vẹn cấu trúc Conv Block từ file 140k của nhóm
    """
    x = layers.Conv2D(
        filters,
        kernel_size=(3, 3),
        padding='same',
        kernel_initializer='he_normal',   # Khởi tạo He normal
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)  # L2 regularization
    )(x)
    x = layers.BatchNormalization()(x)    # Chuẩn hóa Batch
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)  # Giảm chiều không gian
    x = layers.Dropout(dropout_rate)(x)
    return x

def build_mhsa_block(x):
    """
    Trích xuất cấu trúc Multi-Head Self-Attention từ file 140k của nhóm
    """
    # Biến đổi chiều không gian (14, 14, 256) -> (196, 256)
    seq_len = x.shape[1] * x.shape[2]   
    x_reshaped = layers.Reshape((seq_len, 256))(x)  

    # Dropout rate 0.3 trước attention
    x_drop = layers.Dropout(0.3)(x_reshaped)

    # Khối MHSA gồm 4 heads, key_dim=64 theo đúng Table 2 của bài báo
    attn_output = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=64,
        dropout=0.2          # Dropout nội bộ 0.2
    )(x_drop, x_drop)        

    # Residual connection (Kết nối tắt)
    x_residual = layers.Add()([x_reshaped, attn_output])

    # Layer Normalization
    x_norm = layers.LayerNormalization()(x_residual)

    return x_norm

In [34]:
# Lắp ghép bộ khung mạng tuần tự hoàn chỉnh
inputs = layers.Input(shape=(224, 224, 3))

# Trích xuất đặc trưng cục bộ qua 4 khối CNN tuần tự
x = build_cnn_block(inputs, filters=32)   
x = build_cnn_block(x, filters=64)        
x = build_cnn_block(x, filters=128)       
x = build_cnn_block(x, filters=256)       

# Học ngữ cảnh toàn cục qua khối MHSA
x = build_mhsa_block(x)

# Thu gọn chuỗi đặc trưng thành một vector phẳng
x = layers.GlobalAveragePooling1D()(x)

# Lớp Dense phân loại cuối cùng
x = layers.Dense(128)(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(64)(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

# Đầu ra phân loại nhị phân (Sigmoid)
outputs = layers.Dense(1, activation='sigmoid')(x)

# Khởi tạo mô hình trống dành riêng cho mảng CCDDF
model_ccddf = Model(inputs=inputs, outputs=outputs)

# In bảng cấu trúc thông số để kiểm tra tổng tham số (Kỳ vọng: 696,001 tham số)
model_ccddf.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 224, 224,  │        896 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        128 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 224, 224,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 112, 112,  │          0 │ activation_4[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 112, 112,  │          0 │ max_pooling2d_4[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 112, 112,  │     18,496 │ dropout_8[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        256 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 56, 56,    │          0 │ activation_5[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 56, 56,    │          0 │ max_pooling2d_5[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 56, 56,    │     73,856 │ dropout_9[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        512 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_6     │ (None, 28, 28,    │          0 │ activation_6[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 28, 28,    │          0 │ max_pooling2d_6[… │
│ (Dropout)           │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 28, 28,    │    295,168 │ dropout_10[0][0]

 Total params: 696,001 (2.66 MB)

 Trainable params: 694,657 (2.65 MB)

 Non-trainable params: 1,344 (5.25 KB)

In [35]:
import tensorflow.keras.backend as K

def focal_loss(alpha=0.25, gamma=2.0):
    """
    Giữ nguyên hàm Focal Loss tùy biến mà nhóm bạn đã tối ưu theo công thức bài báo
    """
    def loss_fn(y_true, y_pred):
        y_pred = K.clip(y_pred, 1e-7, 1 - 1e-7)  
        bce = -y_true * K.log(y_pred) - (1 - y_true) * K.log(1 - y_pred)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_weight = K.pow(1 - p_t, gamma)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        return K.mean(alpha_t * focal_weight * bce)
    return loss_fn

# BIÊN DỊCH MÔ HÌNH: Ép tốc độ học (learning_rate) về 1e-5 cho CCDDF (Table 3)
model_ccddf.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # <--- Điểm thay đổi cốt lõi
    loss=focal_loss(alpha=0.25, gamma=2.0),
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)
print("Mô hình đã được lấy cấu trúc và biên dịch siêu tham số CCDDF thành công!")

Mô hình đã được lấy cấu trúc và biên dịch siêu tham số CCDDF thành công!


In [39]:
CHECKPOINT_PATH_CCDDF = '/kaggle/working/best_model_ccddf.keras'

callbacks_ccddf = [
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=2, restore_best_weights=True, mode='max', verbose=1),
    tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH_CCDDF, monitor='val_auc', save_best_only=True, mode='max', verbose=1)
]

print("--- BẮT ĐẦU KÍCH HOẠT TIẾN TRÌNH HUẤN LUYỆN CCDDF ---")
# Thực hiện gọi hàm fit từ biến 'model' chuẩn của bạn
history_ccddf = model_ccddf.fit(
    train_gen,                         # Đã được sửa lỗi định nghĩa ở Cell 3
    epochs=10,                         # Chạy đúng 10 chu kỳ theo bài báo
    validation_data=val_gen,           # Đã được sửa lỗi định nghĩa ở Cell 3
    callbacks=callbacks_ccddf,
    verbose=1
)

--- BẮT ĐẦU KÍCH HOẠT TIẾN TRÌNH HUẤN LUYỆN CCDDF ---
Epoch 1/10


I0000 00:00:1780803374.786028     153 service.cc:152] XLA service 0x47205040 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780803374.786097     153 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780803374.786118     153 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780803375.984797     153 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-06-07 03:36:26.570518: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-07 03:36:26.753363: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1780803401.788999     153 device_compil

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5041 - auc: 0.5016 - loss: 0.3359 - precision: 0.5030 - recall: 0.5020
Epoch 1: val_auc improved from None to 0.44789, saving model to /kaggle/working/best_model_ccddf.keras

Epoch 1: finished saving model to /kaggle/working/best_model_ccddf.keras
125/125 ━━━━━━━━━━━━━━━━━━━━ 241s 2s/step - accuracy: 0.5037 - auc: 0.5048 - loss: 0.3178 - precision: 0.5037 - recall: 0.4991 - val_accuracy: 0.5000 - val_auc: 0.4479 - val_loss: 0.1956 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5038 - auc: 0.5118 - loss: 0.2901 - precision: 0.5021 - recall: 0.4958
Epoch 2: val_auc did not improve from 0.44789
125/125 ━━━━━━━━━━━━━━━━━━━━ 197s 2s/step - accuracy: 0.5066 - auc: 0.5124 - loss: 0.2882 - precision: 0.5068 - recall: 0.4965 - val_accuracy: 0.4991 - val_auc: 0.4474 - val_loss: 0.1872 - val_precision: 0.2857 - val_recall: 0.0012
Epoch 3/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2